In [0]:
%sql
use catalog 0725catalog;
drop table if exists 0725catalog.silver.customer_dim;
CREATE TABLE IF NOT EXISTS 0725catalog.silver.customer_dim (
  customerID STRING,
  country STRING,
  start_date DATE,  -- When this version became active
  end_date DATE,    -- When it was superseded (null if current)
  current BOOLEAN   -- Boolean flag for active recor
)
USING DELTA;

-- describe extended 0725catalog.silver.customer_dim;
-- truncate table  0725catalog.silver.customer_dim;
-- VACUUM silver.customer_dim RETAIN 0 HOURS;


In [0]:
dbutils.fs.rm("dbfs:/mnt/volume/path/to/table", recurse=True)

In [0]:
%sql
MERGE INTO silver.customer_dim AS target
USING bronze.customers_raw AS source
ON target.customerID = source.customerID AND target.current = true

WHEN MATCHED AND target.country <> source.country THEN
  UPDATE SET
    target.current = false,
    target.end_date = current_date()

WHEN NOT MATCHED THEN
  INSERT (
    customerID, country, start_date, end_date, current
  )
  VALUES (
    source.customerID, source.country, current_date(), null, true
  )
